# 03-2. Embedding과 Vector Retrieval

- 핵심 기술: Embedding, Vector Store, 유사도 점수 해석, Top-k 검색, Metadata Filter
- 최종 산출물: 유사도 기반 검색 파이프라인과 검색 결과 CSV

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. Embedding이 텍스트를 벡터 공간에 표현하는 방식을 설명할 수 있다.
2. 코사인 유사도 점수의 의미와 한계를 설명하고 검색 순위에 활용할 수 있다.
3. Vector Store의 Top-k 검색과 Metadata Filter를 사용해 검색 파이프라인을 구성할 수 있다.
4. 검색어의 표현 방식(짧음/구체적, 동의어, 동일 키워드의 다른 의미)이 검색 결과에
   미치는 영향을 관찰할 수 있다.

## 2. 문제 상황

Notebook 03-1에서 문서를 Chunk로 나눴다면, 이제 질문이 들어왔을 때 그 많은 Chunk 중
어떤 것을 검색 결과로 돌려줄지 정해야 한다. `search_keyword()`처럼 정확히 같은
단어가 있는지만 확인하는 방식은 "재택근무"로 검색했을 때 "리모트워크"라는 단어를
쓴 문서를 찾지 못한다. Embedding은 텍스트를 벡터로 바꿔, 정확히 같은 단어가 아니어도
의미가 비슷하면 가까운 벡터로 표현되도록 한다. 이 Notebook은 Embedding과 코사인
유사도를 이용한 검색 흐름과 결과 해석 방법을 확인한다.

## 3. 핵심 개념

### 3.1 정의

Embedding은 텍스트를 고정된 차원의 실수 벡터로 변환하는 것이다. 이렇게 만들어진
벡터 공간에서는 의미가 비슷한 텍스트일수록 벡터 사이의 거리가 가깝다.

### 3.2 개념이 필요한 이유

키워드 검색은 문서에 질문과 똑같은 단어가 있어야만 그 문서를 찾을 수 있다.
그러나 사용자는 같은 의미를 다른 단어로 표현하는 경우가 많다("재택근무" vs
"리모트워크"). Embedding 기반 검색은 단어가 달라도 의미가 비슷하면 찾아낼 수 있어,
검색어 표현에 덜 민감한 검색이 가능해진다.

### 3.3 주요 구성요소

| 구성요소 | 의미 |
|---|---|
| Vector Store | 문서 Embedding과 metadata를 저장하고 유사도 검색을 수행하는 구성요소 |
| 문서/질문 Embedding | 문서와 질문을 같은 벡터 공간으로 변환한 결과 |
| 코사인 유사도 | 질문과 문서 벡터가 얼마나 비슷한지를 비교하는 순위 점수 |
| 순위(rank) | 코사인 유사도가 높은 순서대로 매긴 순번 |
| Top-k | 순위가 높은 상위 k개 결과만 선택하는 것 |
| Metadata Filter | `document_id`, `section` 등 조건으로 검색 후보군을 미리 좁히는 것 |

### 3.4 동작 과정

```text
샘플 문서 → Document 변환 → Vector Store에 저장
                                  ↓
질문 → similarity_search_with_relevance_scores(query, k, filter)
                                  ↓
      Vector Store가 Embedding·유사도·정렬·Top-k 처리
                                  ↓
                    검색 결과 DataFrame
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `get_chroma_store()` | 공통 경로에서 디스크 기반 Chroma Collection 생성·재사용 |
| `add_documents_if_empty()` | 기존 Collection이 비었을 때만 문서와 metadata 저장 |
| `similarity_search_with_relevance_scores(...)` | 질문 Embedding·유사도·정렬·Top-k 검색 |
| `make_metadata_filter()` | Vector Store에 전달할 Metadata Filter |
| `search()` | 검색 조건을 Vector Store에 전달하고 결과 형식을 변환 |
| `build_result_dataframe()` | 검색 결과 DataFrame 출력 |

### 3.6 유사 개념과의 차이

**반드시 교정해야 할 오해**

```text
유사도 점수 0.85
≠ 정답일 확률 85%
```

코사인 유사도는 질문 벡터와 문서 벡터가 얼마나 비슷한지를 비교하는 점수일 뿐,
그 문서가 질문에 대한 올바른 답을 담고 있다는 보장이 아니다. 유사도 점수는 순위를
매기는 데 쓰는 상대적 지표이지, 정답일 확률 같은 절대적 지표가 아니다.

**키워드 검색 vs Embedding 검색**

| 구분 | 키워드 검색 | Embedding 검색 |
|---|---|---|
| 판단 기준 | 문자열이 정확히 일치하는가 | 벡터 방향이 얼마나 비슷한가 |
| 동의어 인식 | 불가능 | 가능(모델 성능에 따라 다름) |
| 계산 비용 | 낮음 | Embedding 계산 비용 발생 |
| 결과 해석 | "포함되어 있다/없다"로 명확함 | 점수는 상대적 순위 신호일 뿐 |

### 3.7 사용 시점과 적용 조건

문서량이 많고 사용자가 다양한 표현으로 질문할 가능성이 높다면 Embedding 검색이
유리하다. 반대로 정확한 코드, ID, 고유명사처럼 정확히 일치해야 의미가 있는 검색은
키워드 검색이 더 적합할 수 있다.

### 3.8 한계와 주의사항

- 코사인 유사도 값 자체는 모델과 문서 집합에 따라 상대적이며, 절대적인 기준값으로
  삼을 수 없다.
- Metadata Filter를 너무 좁게 걸면 실제 정답이 담긴 문서가 후보군에서 아예
  제외될 수 있다.
- Top-k가 너무 작으면 관련 문서를 놓치고, 너무 크면 무관한 문서까지 함께
  전달되어 비용이 늘어난다.

### 3.9 자주 발생하는 오해

"유사도 점수가 높으면 그 문서가 정답"이라는 오해가 가장 흔하다. 실제로는 유사도
점수가 높아도 질문과 무관한 이유로 벡터가 가까워졌을 수 있고, 반대로 진짜 정답을
담은 문서의 점수가 더 낮게 나올 수도 있다. 유사도 점수는 후보를 좁히는 신호일
뿐이며, 실제로 정답인지는 문서 본문을 확인해야 한다.

### 3.10 핵심 정리

- Embedding은 텍스트를 벡터로 표현해 의미 기반 검색을 가능하게 한다.
- Vector Store가 Embedding 저장, 유사도 계산, 정렬, Top-k를 처리하므로 애플리케이션에서
  같은 기능을 다시 구현할 필요가 없다.
- 유사도 점수는 검색 순위를 위한 상대적 지표이며, 정답일 확률이 아니다.
- Metadata Filter는 유사도 검색 전에 후보군 자체를 좁히는 절차다.
- 검색 결과의 실제 정확성은 유사도 점수가 아니라 문서 본문을 확인해야 판단할 수
  있다.

## 4. 실행 구조

```text
SAMPLE_DOCUMENTS → Document 변환
   │
   └─ vector_store.add_documents()
            │
            └─ search(query, k, metadata)
                  └─ vector_store.similarity_search_with_relevance_scores()
                        ├─ Metadata Filter
                        ├─ 질문 Embedding과 유사도 검색
                        └─ 정렬된 Top-k 반환
               ↓
      build_result_dataframe() → 검색 결과 표
               ↓
      outputs/retrieval/retrieval_results.csv 저장
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [1]:
from langchain_core.documents import Document

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_embedding_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import CHROMA_DIR, OUTPUT_DIR
from agentic_ai.retrieval_utils import add_documents_if_empty, get_chroma_store
from agentic_ai.tools import search_keyword

settings = get_settings()
print_environment_summary(settings, needs_embedding_model=True)


[환경 설정 확인]
- 프로젝트: E:\agentic_ai_lab
- 데이터: E:\agentic_ai_lab\data
- 출력: E:\agentic_ai_lab\outputs
- OPENAI_API_KEY: 설정됨
- Embedding Model: text-embedding-3-small


## 6. 유사도 점수 먼저 읽기

이 실습에서는 코사인 유사도 공식을 직접 구현하지 않는다. 검색 결과에서 점수가 클수록
질문과 문서가 상대적으로 더 유사해 높은 순위를 받는다는 점에 집중한다. 아래 예시처럼
같은 후보군 안에서 점수를 비교해 순서를 정하며, 점수 자체는 정답 확률을 의미하지 않는다.

In [2]:
score_examples = [
    {"후보": "문서 A", "유사도": 0.71},
    {"후보": "문서 B", "유사도": 0.92},
    {"후보": "문서 C", "유사도": 0.34},
]

ranked_examples = sorted(score_examples, key=lambda item: item["유사도"], reverse=True)
for rank, example in enumerate(ranked_examples, start=1):
    print(f"{rank}위 {example['후보']}: {example['유사도']:.2f}")

print("주의: 가장 높은 점수도 '정답일 확률'은 아닙니다.")

1위 문서 B: 0.92
2위 문서 A: 0.71
3위 문서 C: 0.34
주의: 가장 높은 점수도 '정답일 확률'은 아닙니다.


## 7. 단계별 구현

### 7.1 샘플 문서 생성

Notebook 03-1의 보고서 내용을 바탕으로 서로 다른 주제를 가진 문서 9개를 만든다.
`report-01`은 원본 보고서, `guide-01`과 `notice-01`은 이후 Metadata Filter와
동의어/동일 키워드 실험에 사용할 별도 문서다.

In [3]:
SAMPLE_DOCUMENTS = [
    {"doc_id": "d1", "document_id": "report-01", "section": "개요",
     "text": "본 보고서는 2025년 한 해 동안 시행된 리모트워크 제도의 운영 현황을 정리하고, 2026년도 제도 개선 방향을 제시하기 위해 작성되었다."},
    {"doc_id": "d2", "document_id": "report-01", "section": "운영 현황",
     "text": "2025년 기준 전체 임직원의 62%가 주 2회 이상 리모트워크를 사용하였다. 부서별로는 개발팀의 사용률이 81%로 가장 높았고, 영업팀은 24%로 가장 낮았다."},
    {"doc_id": "d3", "document_id": "report-01", "section": "만족도 조사 결과",
     "text": "전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다. 불만족 사유로는 협업 지연, 화상회의 피로도, 장비 지원 부족 순으로 나타났다."},
    {"doc_id": "d4", "document_id": "report-01", "section": "문제점",
     "text": "일부 부서에서 리모트워크와 사무실 근무 인원 간 정보 격차가 발생하였다. 특히 신입 직원의 온보딩 과정에서 어려움을 겪는 사례가 보고되었다."},
    {"doc_id": "d5", "document_id": "report-01", "section": "개선 방향",
     "text": "2026년에는 팀별 필수 출근일을 지정하고, 화상회의 시간을 기본 30분으로 단축하는 가이드라인을 도입할 예정이다."},
    {"doc_id": "d6", "document_id": "report-01", "section": "결론",
     "text": "리모트워크 제도는 임직원 만족도와 생산성 측면에서 긍정적인 효과를 보였으나, 협업과 온보딩 절차에 대한 보완이 필요하다."},
    {"doc_id": "d7", "document_id": "report-01", "section": "임원 반응",
     "text": "일부 임원진은 신규 리모트워크 정책 확대에 회의적인 시각을 보였다."},
    {"doc_id": "d8", "document_id": "guide-01", "section": "재택근무 가이드",
     "text": "재택근무를 하는 직원은 매일 오전 정기 화상 회의에 접속해 업무 현황을 공유해야 한다."},
    {"doc_id": "d9", "document_id": "notice-01", "section": "사내 동호회",
     "text": "이번 분기 사내 등산 동호회 정기 모임은 다음 달 첫째 주 토요일에 진행된다."},
]
print(f"샘플 문서 수: {len(SAMPLE_DOCUMENTS)}")

샘플 문서 수: 9


### 7.2 Vector Store 구성

문서를 `Document`로 변환해 Vector Store에 추가한다. 문서 Embedding 생성과 저장은
Vector Store가 담당하며, 애플리케이션에서 Embedding 목록을 직접 관리하지 않는다.

> 이 실습은 로컬 디스크의 `outputs/vectorstore/chroma/`에 Chroma DB를 저장한다.
> 노트북을 다시 실행해도 같은 컬렉션과 문서를 재사용하며, 동일한 문서 ID는 새로
> 중복 추가하지 않고 갱신한다.

In [4]:
embedding_model = get_embedding_model()
# 검색 대상 본문은 page_content에, 필터·출처 정보는 metadata에 분리해 저장한다.
VECTOR_DOCUMENTS = [
    Document(
        page_content=record["text"],
        metadata={key: value for key, value in record.items() if key != "text"},
    )
    for record in SAMPLE_DOCUMENTS
]

COLLECTION_NAME = "sample_documents"
vector_store = get_chroma_store(
    COLLECTION_NAME,
    embedding_model=embedding_model,
)
# 영속 Collection이 비어 있을 때만 추가해 Notebook 재실행 시 중복 적재를 막는다.
documents_added, document_count = add_documents_if_empty(
    vector_store,
    VECTOR_DOCUMENTS,
    ids=[record["doc_id"] for record in SAMPLE_DOCUMENTS],
)

action = "초기화" if documents_added else "기존 Collection 재사용"
print(f"Chroma Collection: {COLLECTION_NAME}")
print(f"디스크 저장 경로: {CHROMA_DIR}")
print(f"처리 결과: {action} ({document_count}개 문서)")


Chroma Collection: sample_documents
디스크 저장 경로: E:\agentic_ai_lab\outputs\vectorstore\chroma
처리 결과: 기존 Collection 재사용 (9개 문서)


### 7.3 유사도 검색

질문을 문자열로 전달하면 Vector Store가 질문 Embedding, 유사도 계산, 정렬, Top-k
선택을 처리한다. 반환값은 `(Document, score)` 튜플 목록이다.

In [5]:
# relevance score는 후보 간 순위를 위한 값이며 정답일 확률로 해석하면 안 된다.
sample_matches = vector_store.similarity_search_with_relevance_scores(
    "리모트워크 만족도는 어땠나요?",
    k=3,
)
for document, score in sample_matches:
    print(f"score={score:.4f} | {document.metadata['section']} | {document.page_content}")

score=0.5313 | 결론 | 리모트워크 제도는 임직원 만족도와 생산성 측면에서 긍정적인 효과를 보였으나, 협업과 온보딩 절차에 대한 보완이 필요하다.
score=0.5001 | 만족도 조사 결과 | 전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다. 불만족 사유로는 협업 지연, 화상회의 피로도, 장비 지원 부족 순으로 나타났다.
score=0.4372 | 운영 현황 | 2025년 기준 전체 임직원의 62%가 주 2회 이상 리모트워크를 사용하였다. 부서별로는 개발팀의 사용률이 81%로 가장 높았고, 영업팀은 24%로 가장 낮았다.


### 7.4 검색 결과 변환

Vector Store가 반환한 `Document`와 점수를 실습에서 관찰하기 쉬운 dict 형태로
변환하고, 반환 순서대로 순위를 부여한다.

In [6]:
def to_result_records(matches: list[tuple[Document, float]]) -> list[dict]:
    """Vector Store 검색 결과를 순위가 포함된 dict 목록으로 변환한다."""
    # Vector Store가 유사도순으로 반환한 순서를 유지하며 1부터 rank를 붙인다.
    return [
        {
            **document.metadata,
            "text": document.page_content,
            "similarity": score,
            "rank": rank,
        }
        for rank, (document, score) in enumerate(matches, start=1)
    ]

### 7.5 Metadata Filter

Chroma의 `filter` 인자에 전달할 metadata 조건 dict를 만든다. 필터는 유사도 검색 전에
DB 내부 후보 문서를 좁히며, 점수 계산과 정렬은 Chroma가 담당한다.

In [7]:
def make_metadata_filter(
    document_id: str | None = None,
    section: str | None = None,
) -> dict | None:
    """Chroma에 전달할 metadata 필터를 만든다."""
    # 선택된 조건만 모아 필터가 없는 경우와 단일·복합 조건을 구분한다.
    conditions = []
    if document_id is not None:
        conditions.append({"document_id": document_id})
    if section is not None:
        conditions.append({"section": section})

    if not conditions:
        return None
    if len(conditions) == 1:
        return conditions[0]
    # 여러 metadata 조건은 모두 만족해야 하므로 Chroma의 $and 연산자로 묶는다.
    return {"$and": conditions}

### 7.6 Vector Store 검색 함수

`search()`는 조건과 `k`를 Vector Store에 전달하고 결과 형식만 변환한다.
Embedding 생성, 유사도 계산, 정렬, Top-k 선택은 직접 구현하지 않는다.

In [8]:
def search(
    query: str,
    k: int = 3,
    document_id: str | None = None,
    section: str | None = None,
) -> list[dict]:
    """Vector Store의 유사도 검색과 Metadata Filter를 실행한다."""
    if k < 1:
        raise ValueError("k는 1 이상이어야 합니다.")
    # 필터는 검색 후 결과를 지우는 것이 아니라 유사도를 계산할 후보군 자체를 제한한다.
    metadata_filter = make_metadata_filter(document_id=document_id, section=section)
    matches = vector_store.similarity_search_with_relevance_scores(
        query,
        k=k,
        filter=metadata_filter,
    )
    return to_result_records(matches)

### 7.7 Metadata Filter 실행

`document_id` 조건을 Vector Store 검색에 전달해 후보 문서가 실제로 좁혀지는지
확인한다.

In [9]:
filtered_demo = search(
    "리모트워크",
    k=len(SAMPLE_DOCUMENTS),
    document_id="report-01",
)
print(f"document_id='report-01' 필터 적용 후 결과 수: {len(filtered_demo)} / 전체 {len(SAMPLE_DOCUMENTS)}")
print({result["document_id"] for result in filtered_demo})

document_id='report-01' 필터 적용 후 결과 수: 7 / 전체 9
{'report-01'}


### 7.8 검색 결과 DataFrame 출력

Vector Store 검색 결과를 DataFrame으로 변환해 순위, metadata, 유사도 점수와 본문을
한눈에 비교한다.

In [10]:
import pandas as pd
def build_result_dataframe(query: str, results: list[dict]) -> pd.DataFrame:
    """검색 결과를 표 형식으로 만든다."""
    rows = [
        {
            "query": query,
            "rank": r["rank"],
            "doc_id": r["doc_id"],
            "document_id": r["document_id"],
            "section": r["section"],
            "similarity": round(r["similarity"], 4),
            "text": r["text"],
        }
        for r in results
    ]
    return pd.DataFrame(rows)


baseline_results = search("리모트워크 만족도는 어땠나요?", k=3)
build_result_dataframe("리모트워크 만족도는 어땠나요?", baseline_results)

,query,rank,doc_id,document_id,section,similarity,text
0,리모트워크 만족도는 어땠나요?,1,d6,report-01,결론,0.5313,"리모트워크 제도는 임직원 만족도와 생산성 측면에서 긍정적인 효과를 보였으나, 협업과..."
1,리모트워크 만족도는 어땠나요?,2,d3,report-01,만족도 조사 결과,0.5001,전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다. 불만족 사유로는 협...
2,리모트워크 만족도는 어땠나요?,3,d2,report-01,운영 현황,0.4372,2025년 기준 전체 임직원의 62%가 주 2회 이상 리모트워크를 사용하였다. 부서...


## 8. 실행 결과 관찰

`baseline_results`의 순위와 유사도 점수, 그리고 실제 본문을 함께 확인한다.

In [11]:
for r in baseline_results:
    print(f"[{r['rank']}위] similarity={r['similarity']:.4f} | section={r['section']} | {r['text']}")

[1위] similarity=0.5313 | section=결론 | 리모트워크 제도는 임직원 만족도와 생산성 측면에서 긍정적인 효과를 보였으나, 협업과 온보딩 절차에 대한 보완이 필요하다.
[2위] similarity=0.5001 | section=만족도 조사 결과 | 전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다. 불만족 사유로는 협업 지연, 화상회의 피로도, 장비 지원 부족 순으로 나타났다.
[3위] similarity=0.4372 | section=운영 현황 | 2025년 기준 전체 임직원의 62%가 주 2회 이상 리모트워크를 사용하였다. 부서별로는 개발팀의 사용률이 81%로 가장 높았고, 영업팀은 24%로 가장 낮았다.


**결과 해석**: 순위(`rank`)는 유사도 점수(`similarity`)를 기준으로 매겨진다. 점수
자체의 절대값보다는, 어떤 문서가 다른 문서보다 상대적으로 질문과 가까운지를
비교하는 데 의미가 있다.

## 9. 비교 실험

### 9.1 Top-1과 Top-3

In [12]:
query_a = "화상회의 피로도를 줄이는 방법은?"
top1 = search(query_a, k=1)
top3 = search(query_a, k=3)
print("Top-1:")
for r in top1:
    print(f"  [{r['rank']}위] {r['section']} | {r['text']}")
print("Top-3:")
for r in top3:
    print(f"  [{r['rank']}위] {r['section']} | {r['text']}")

Top-1:
  [1위] 개선 방향 | 2026년에는 팀별 필수 출근일을 지정하고, 화상회의 시간을 기본 30분으로 단축하는 가이드라인을 도입할 예정이다.
Top-3:
  [1위] 개선 방향 | 2026년에는 팀별 필수 출근일을 지정하고, 화상회의 시간을 기본 30분으로 단축하는 가이드라인을 도입할 예정이다.
  [2위] 재택근무 가이드 | 재택근무를 하는 직원은 매일 오전 정기 화상 회의에 접속해 업무 현황을 공유해야 한다.
  [3위] 만족도 조사 결과 | 전체 응답자의 74%가 리모트워크 제도에 만족한다고 응답하였다. 불만족 사유로는 협업 지연, 화상회의 피로도, 장비 지원 부족 순으로 나타났다.


**관찰**: Top-1만 보면 가장 유사도가 높은 문서 하나만 확인할 수 있지만, 실제 정답에
필요한 정보가 2~3위 문서에 나뉘어 있을 수도 있다. Top-k를 늘리면 놓칠 수 있는 정보를
줄일 수 있지만, 그만큼 무관한 문서가 섞일 가능성도 커진다.

### 9.2 짧은 검색어와 구체적인 검색어

In [13]:
# 같은 주제라도 구체적인 검색어가 더 많은 의미 단서를 제공해 순위를 바꿀 수 있다.
short_query = "회의"
specific_query = "화상회의 피로도를 낮추기 위한 회의 시간 단축 방안"

short_results = search(short_query, k=3)
specific_results = search(specific_query, k=3)

print(f"짧은 검색어 '{short_query}':")
for r in short_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | {r['section']}")
print(f"구체적인 검색어 '{specific_query}':")
for r in specific_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | {r['section']}")

짧은 검색어 '회의':
  [1위] similarity=0.2972 | 재택근무 가이드
  [2위] similarity=0.2907 | 사내 동호회
  [3위] similarity=0.2585 | 개선 방향
구체적인 검색어 '화상회의 피로도를 낮추기 위한 회의 시간 단축 방안':
  [1위] similarity=0.4935 | 개선 방향
  [2위] similarity=0.4153 | 재택근무 가이드
  [3위] similarity=0.3610 | 임원 반응


**관찰**: 짧은 검색어는 여러 문서와 폭넓게 비슷하게 나올 수 있어 순위 사이의 점수
차이가 작을 수 있다. 구체적인 검색어는 관련 문서와 무관한 문서 사이의 유사도 차이가
더 뚜렷하게 나타나는 경향이 있다.

### 9.3 동의어

In [14]:
original_term_query = "리모트워크 직원의 화상 회의 참여 방식"
synonym_query = "재택근무 직원의 화상 회의 참여 방식"

original_results = search(original_term_query, k=3)
synonym_results = search(synonym_query, k=3)

print("'리모트워크' 검색 결과:")
for r in original_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | doc_id={r['doc_id']} | {r['section']}")
print("'재택근무'(동의어) 검색 결과:")
for r in synonym_results:
    print(f"  [{r['rank']}위] similarity={r['similarity']:.4f} | doc_id={r['doc_id']} | {r['section']}")

'리모트워크' 검색 결과:
  [1위] similarity=0.5207 | doc_id=d6 | 결론
  [2위] similarity=0.4780 | doc_id=d2 | 운영 현황
  [3위] similarity=0.4661 | doc_id=d8 | 재택근무 가이드
'재택근무'(동의어) 검색 결과:
  [1위] similarity=0.7281 | doc_id=d8 | 재택근무 가이드
  [2위] similarity=0.3871 | doc_id=d2 | 운영 현황
  [3위] similarity=0.3811 | doc_id=d3 | 만족도 조사 결과


**관찰**: `search_keyword()` 같은 키워드 검색은 "재택근무"라는 단어가 없는 문서(d1~d7)를
찾지 못한다. 실제 실행 결과를 보면 d8(재택근무 가이드)은 "리모트워크" 검색에서는
3위(similarity=0.4661)에 그치지만, "재택근무"로 검색하면 1위(similarity=0.7281)로
올라온다. "재택근무"라는 단어가 문서에 없어도(리모트워크 검색 시) Embedding 검색은
이 문서를 상위권에서 찾아내며, 검색어를 동의어로 바꾸면 그 순위가 더 뚜렷하게
올라간다는 것을 확인할 수 있다.

### 9.4 동일 키워드의 다른 의미

In [15]:
# 키워드 검색은 표면 문자열만 보지만 임베딩 검색은 질문 문맥으로 두 의미를 구분한다.
d3_text = SAMPLE_DOCUMENTS[2]["text"]  # 화상회의 피로도(회의 = 모임)
d7_text = SAMPLE_DOCUMENTS[6]["text"]  # 회의적인 시각(회의 = 의심)

keyword_hit_d3 = search_keyword(d3_text, "회의")
keyword_hit_d7 = search_keyword(d7_text, "회의")
print("키워드 검색('회의') 결과:")
print(f"  d3(화상회의 피로도): found={keyword_hit_d3['found']}")
print(f"  d7(회의적인 시각):   found={keyword_hit_d7['found']}")

meeting_query = "회의 때문에 피곤한 이유"
meeting_results = search(meeting_query, k=len(SAMPLE_DOCUMENTS))
d3_rank = next(r["rank"] for r in meeting_results if r["doc_id"] == "d3")
d7_rank = next(r["rank"] for r in meeting_results if r["doc_id"] == "d7")
print(f"'{meeting_query}' 질문에서 d3 순위: {d3_rank}, d7 순위: {d7_rank}")

키워드 검색('회의') 결과:
  d3(화상회의 피로도): found=True
  d7(회의적인 시각):   found=True
'회의 때문에 피곤한 이유' 질문에서 d3 순위: 1, d7 순위: 6


**관찰**: 키워드 검색은 "회의"라는 글자만 보고 d3(화상회의)와 d7(회의적인 시각) 모두
"찾음"으로 판단한다. 그러나 두 문서에서 "회의"는 전혀 다른 의미(모임 vs 의심)로
쓰였다. Embedding 검색이 문맥을 제대로 반영한다면, 실제로 회의(모임)에 관해 묻는
질문에서는 d3의 순위가 d7보다 높게 나와야 한다.

### 9.5 Metadata Filter 적용 전후

In [16]:
query_b = "정기 모임 일정이 궁금해요"
# 동일한 query라도 metadata filter가 검색 가능한 문서 범위를 바꾼다.
without_filter = search(query_b, k=3)
with_filter = search(query_b, k=3, document_id="report-01")

print("Filter 미적용 (전체 9개 문서가 후보):")
for r in without_filter:
    print(f"  [{r['rank']}위] document_id={r['document_id']} | {r['section']}")
print("Filter 적용 (document_id='report-01'만 후보, 7개):")
for r in with_filter:
    print(f"  [{r['rank']}위] document_id={r['document_id']} | {r['section']}")

Filter 미적용 (전체 9개 문서가 후보):
  [1위] document_id=guide-01 | 재택근무 가이드
  [2위] document_id=report-01 | 임원 반응
  [3위] document_id=report-01 | 개선 방향
Filter 적용 (document_id='report-01'만 후보, 7개):
  [1위] document_id=report-01 | 임원 반응
  [2위] document_id=report-01 | 개선 방향
  [3위] document_id=report-01 | 만족도 조사 결과


**관찰**: Filter 없이 검색하면 `document_id`가 다른 d8(guide-01, 재택근무 가이드)이
1위로 섞여 들어온다. "정기 모임 일정"이라는 질문의 실제 관련 문서는 아니지만, 유사도
계산 시 후보에서 제외되지 않았기 때문이다. `document_id`를 `"report-01"`로 제한하면
d8이 애초에 후보군에서 빠져, 원래 2~3위였던 report-01 문서(임원 반응, 개선 방향)가
1~2위로 올라오고 그 아래 순위였던 문서가 새로 3위에 나타난다. Metadata Filter는
유사도를 계산하기 전에 후보군 자체를 바꾼다는 점이 핵심이다.

## 10. 실패 실험과 교정: 유사도 점수를 정답 확률로 오해하기

실제 문서 대신 통제된 검색 결과를 사용해, Vector Store가 반환한 유사도 점수가 높아도
그 문서가 질문의 정답을 담고 있다고 보장할 수 없음을 확인한다.

In [17]:
controlled_results = [
    {"doc_id": "A", "similarity": 0.95, "contains_answer": False},
    {"doc_id": "B", "similarity": 0.78, "contains_answer": True},
]

top_result = max(controlled_results, key=lambda item: item["similarity"])
print("가장 높은 점수의 문서:", top_result)
print("실제 정답 포함 여부:", top_result["contains_answer"])
print("→ 유사도 점수는 후보 순위를 정할 뿐, 정답 여부는 본문이나 근거 평가로 확인해야 한다.")

가장 높은 점수의 문서: {'doc_id': 'A', 'similarity': 0.95, 'contains_answer': False}
실제 정답 포함 여부: False
→ 유사도 점수는 후보 순위를 정할 뿐, 정답 여부는 본문이나 근거 평가로 확인해야 한다.


**교정된 접근**: 유사도 점수를 "정답 확률"로 취급해 자동으로 채택하지 않는다. 대신
Top-k로 후보를 좁힌 뒤, 각 후보의 실제 본문을 확인하거나(사람 또는 LLM), Notebook 03-3에서
다룰 "근거 적합성 평가" 단계를 거쳐 정말로 질문에 답할 수 있는 내용인지 검증해야 한다.
8~9번에서 얻은 검색 결과도 순위와 점수만 보지 말고, `text` 필드를 직접 읽어 질문에
대한 답이 맞는지 확인하는 습관이 필요하다.

## 11. 도전 과제

1. `SAMPLE_DOCUMENTS`에 완전히 새로운 주제의 문서를 2~3개 추가하고, 관련 없는
   질문을 던졌을 때 유사도 점수가 어떻게 분포하는지 관찰한다.
2. `search()`에 `section` 필터까지 함께 적용해, `document_id`와 `section`을 동시에
   좁혔을 때 후보 수가 어떻게 줄어드는지 확인한다.
3. Top-k를 1부터 9까지 바꿔가며 같은 질문을 검색하고, 몇 번째 순위부터 유사도
   점수가 급격히 낮아지는지(문서와 무관해지는지) 관찰한다.

## 12. 테스트

**테스트 유형: 외부 API 통합 테스트 — OpenAI Embedding + Chroma**

검색 순위와 metadata filter를 검증한다. 실패하면 코드와 함께 API Key, 네트워크, Embedding 모델과 Chroma Collection 상태를 확인한다.

In [18]:
search_check = search("리모트워크 만족도", k=3, document_id="report-01")
assert len(search_check) == 3
assert [result["rank"] for result in search_check] == [1, 2, 3]
assert all(
    search_check[index]["similarity"] >= search_check[index + 1]["similarity"]
    for index in range(len(search_check) - 1)
)
assert all(r["document_id"] == "report-01" for r in search_check)

all_filtered_check = search("리모트워크", k=len(SAMPLE_DOCUMENTS), document_id="report-01")
assert len(all_filtered_check) == 7
assert all(r["document_id"] == "report-01" for r in all_filtered_check)

try:
    search("리모트워크", k=0)
    raise AssertionError("k=0이 통과했습니다.")
except ValueError:
    pass

print("테스트 통과")

테스트 통과


## 13. 결과 저장

In [19]:
all_result_rows = []
for label, results in [
    ("baseline", baseline_results),
    ("top1_top3_top1", top1),
    ("top1_top3_top3", top3),
    ("short_query", short_results),
    ("specific_query", specific_results),
    ("original_term", original_results),
    ("synonym", synonym_results),
    ("without_filter", without_filter),
    ("with_filter", with_filter),
]:
    df = build_result_dataframe(label, results)
    all_result_rows.extend(df.to_dict(orient="records"))

retrieval_results_df = pd.DataFrame(all_result_rows)
retrieval_dir = OUTPUT_DIR / "retrieval"
retrieval_dir.mkdir(parents=True, exist_ok=True)
retrieval_csv_path = retrieval_dir / "retrieval_results.csv"
retrieval_results_df.to_csv(retrieval_csv_path, index=False, encoding="utf-8-sig")
print("저장 위치:", retrieval_csv_path)

embedding_log = {
    "document_count": len(SAMPLE_DOCUMENTS),
    "vector_store": type(vector_store).__name__,
    "baseline_query": "리모트워크 만족도는 어땠나요?",
    "baseline_top_results": [
        {"rank": r["rank"], "doc_id": r["doc_id"], "similarity": round(r["similarity"], 4)}
        for r in baseline_results
    ],
}
saved_path = save_log(embedding_log, OUTPUT_DIR / "logs" / "03-2_embedding_retrieval_log.json")
print("저장 위치:", saved_path)

저장 위치: E:\agentic_ai_lab\outputs\retrieval\retrieval_results.csv
저장 위치: E:\agentic_ai_lab\outputs\logs\03-2_embedding_retrieval_log.json


## 14. 핵심 정리

- Embedding은 텍스트를 벡터로 표현해 의미 기반 검색을 가능하게 한다.
- Vector Store는 문서 Embedding 저장, 질문 Embedding, 유사도 계산, 정렬, Top-k
  선택을 담당하며 애플리케이션은 검색 조건과 결과 활용에 집중한다.
- 유사도 점수는 순위를 매기는 상대적 신호일 뿐, 정답일 확률이 아니다.
- `k`는 Vector Store가 반환할 상위 결과 수를 정하고, Metadata Filter는 유사도 검색
  전에 후보군 자체를 좁힌다.
- 짧은 검색어, 동의어, 동일 키워드의 다른 의미는 모두 검색 결과에 영향을 주며,
  키워드 검색과 Embedding 검색이 서로 다르게 반응한다.
- 검색 결과를 그대로 신뢰하지 않고, 실제 본문을 확인해 정말 질문에 답이 되는지
  검증하는 절차가 필요하다.

## 15. 확인 문제

1. 유사도 점수 0.85가 "정답일 확률 85%"를 의미하지 않는 이유는 무엇인가?
2. 순위 정렬/Top-k와 Metadata Filter는 검색 후보군에 어떻게 다르게 영향을 주는가?
3. 키워드 검색이 "재택근무"와 "리모트워크"를 같은 의미로 찾지 못하는 이유는
   무엇인가?
4. 유사도 점수가 높은 검색 결과를 받았을 때, 그것을 바로 정답으로 채택하면 안 되는
   이유는 무엇이며 대신 어떻게 해야 하는가?